In [16]:
from langgraph.graph import StateGraph, START, END
from openai import OpenAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [17]:
load_dotenv()

True

In [18]:
model = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

In [19]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [20]:
def create_outline(state: BlogState) -> BlogState:
    # Fetch title
    title = state["title"]

    # Call LLM to create outline
    prompt = f"Create an outline for a blog on the topic of: {title}"
    response = model.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="openai/gpt-oss-120b",
    ).choices[0].message.content

    # Update state with outline
    state["outline"] = response
    
    return state

In [21]:
def create_blog(state: BlogState) -> BlogState:
    # Fetch title and outline
    title = state["title"]
    outline = state["outline"]

    # Call LLM to create blog content
    prompt = f"Write a blog post with the title '{title}' using the following outline:\n{outline}"
    response = model.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="openai/gpt-oss-120b",
    ).choices[0].message.content

    # Update state with content
    state["content"] = response
    
    return state

In [22]:
graph = StateGraph(BlogState)

In [23]:
graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)

In [24]:
graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", END)

In [25]:
workflow = graph.compile()

In [26]:
initial_state = {"title": "LangGraph: A key concept for automating LLM workflows"}

final_state = workflow.invoke(initial_state)

print("Final State:")
print(final_state)

Final State:
{'title': 'LangGraph: A key concept for automating LLM workflows', 'outline': '**Blog Outline – “LangGraph: A Key Concept for Automating LLM Workflows”**  \n\n---\n\n### 1. Introduction  \n   - **Hook** – A short anecdote or statistic about the growing complexity of LLM pipelines (e.g., “From prompt‑engineering to data‑augmentation, today’s LLM projects involve 10+ moving parts”).  \n   - **Why automation matters** – Speed, reproducibility, cost‑efficiency, and the need to scale across teams.  \n   - **Enter LangGraph** – One‑sentence definition and promise.  \n   - **What readers will learn** – Brief bullet list of take‑aways (conceptual model, core primitives, real‑world patterns, how to start building).  \n\n### 2. The Problem Space: Manual LLM Orchestration  \n   - **Typical workflow fragments**  \n     - Prompt design & chaining  \n     - Data preprocessing & augmentation  \n     - Model selection, fine‑tuning, inference  \n     - Post‑processing, filtering, routing r

In [28]:
print(final_state["content"])

# LangGraph: A Key Concept for Automating LLM Workflows  

*By [Your Name]* – *March 7 2026*  

---  

## 1. Introduction  

> **“From prompt‑engineering to data‑augmentation, today’s LLM projects involve more than 10 moving parts.”**  

If you’ve ever built a Retrieval‑Augmented Generation (RAG) service, you’ve probably written a handful of scripts that juggle prompt templates, vector‑store look‑ups, post‑processing filters, and occasional API calls. The code quickly morphs into a tangled web of imports, global variables, and “magic” side‑effects.  

### Why Automation Matters  

| Benefit | What it means for LLM teams |
|--------|-----------------------------|
| **Speed** | Spin up new pipelines in minutes, not days. |
| **Reproducibility** | Every run is versioned, every token count logged. |
| **Cost‑efficiency** | Skip expensive model calls when confidence is high. |
| **Scalability** | Deploy the same workflow across dozens of micro‑services or data‑science notebooks. |

### Ente